# ParsingAI — Colab 版
依序執行每個 Cell。

In [ ]:
# ── Cell 1: Clone repo & 安裝套件 ─────────────────────────────────────────────
!git clone https://github.com/410773004/parsingAI.git /content/parsingAI -q
!pip install ollama tiktoken -q

import sys
sys.path.insert(0, '/content/parsingAI')
print('✓ 完成')

In [ ]:
# ── Cell 2: 安裝 Ollama & 拉模型（約 2.5GB，需等幾分鐘）────────────────────────
!apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

from settings import config
print(f'拉取模型：{config.MODEL}')
!ollama pull {config.MODEL}
print('✓ 完成')

In [ ]:
# ── Cell 3: 上傳 log 資料夾的 ZIP 檔 ──────────────────────────────────────────
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()  # 選擇 ZIP 檔
zip_name = list(uploaded.keys())[0]
folder_name = Path(zip_name).stem

extract_dir = Path(f'/content/{folder_name}')
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    # 若 ZIP 內有同名子資料夾，直接使用該子資料夾
    members = z.namelist()
    top_dirs = {m.split('/')[0] for m in members if '/' in m}
    if len(top_dirs) == 1 and list(top_dirs)[0] == folder_name:
        z.extractall('/content/')
    else:
        z.extractall(extract_dir)

log_files = list(extract_dir.glob('*.log'))
print(f'解壓完成：{extract_dir}')
print(f'找到 {len(log_files)} 個 .log 檔案')

In [ ]:
# ── Cell 4: 執行分析 ───────────────────────────────────────────────────────────
# 修改下面這行描述問題
issue = "請輸入問題描述"

from services.analyzer import analyze

print('分析中，請稍候...')
result = analyze(extract_dir, issue, folder_name)

print(f"\nProject    : {result['project']}")
print(f"FW Version : {result['fw_version']}")
print(f"Serial     : {result['serial']}")
print(f"Token 數   : {result['token_count']}")
print('\n' + '=' * 80)
print(result['llm_output'])

In [ ]:
# ── Cell 5: 儲存結果到檔案（選用）────────────────────────────────────────────
output_path = Path(f'/content/{folder_name}_result.txt')
output_path.write_text(
    f"Project    : {result['project']}\n"
    f"FW Version : {result['fw_version']}\n"
    f"Serial     : {result['serial']}\n"
    f"Token 數   : {result['token_count']}\n"
    f"{'=' * 80}\n"
    f"{result['llm_output']}\n"
    f"{'=' * 80}\n"
    f"{result['cleaned_log']}",
    encoding='utf-8'
)

files.download(str(output_path))
print(f'已下載：{output_path.name}')